<a href="https://colab.research.google.com/github/vhvaldiviesor/Programacion-para-analitica-descriptiva-y-predictiva/blob/main/Sesion10_269711.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Sesión 10 — Entregable sobre Data Profiling

**Nombre completo:** Victor Hugo Valdivieso

**Matrícula:** 269711

---

Este notebook contiene las actividades a entregar de la Sesión 10 (Data Profiling), aplicadas sobre **datasets reales**: el catálogo de Netflix y el dataset de Customer Personality Analysis (Kaggle). Si aún no revisaste las explicaciones y ejemplos de cada tema, hazlo primero en el notebook `Sesion10_Data_Profiling_Actividad_Asincrona.ipynb`.

**Nota sobre los datos:** ambos datasets son reales — los problemas de calidad que vas a encontrar (nulos, categorías inconsistentes, valores fuera de rango) ya existían antes de que este notebook los usara. La única excepción está marcada explícitamente en la Actividad 3 y la Práctica integradora, donde se inyectan un par de filas/valores a propósito para poder practicar duplicados y ajuste de tipos con un resultado garantizado.

**Antes de entregar:** ejecuta "Reiniciar y ejecutar todo" para confirmar que tu notebook corre de principio a fin sin errores.

## Preparación

Ejecuta esta celda antes de empezar — descarga los dos datasets reales que vas a usar.

In [ ]:
import pandas as pd

url_netflix = 'https://raw.githubusercontent.com/Vibe1990/Netflix-Project/main/netflix_title.csv'
url_marketing = 'https://raw.githubusercontent.com/amankharwal/Website-data/master/marketing_campaign.csv'

df_netflix = pd.read_csv(url_netflix)
df_marketing = pd.read_csv(url_marketing, sep=';')  # nota: este archivo usa punto y coma, no coma

print('Netflix:', df_netflix.shape)
print('Marketing:', df_marketing.shape)

Netflix: (7787, 12)
Marketing: (2240, 29)


---
## Actividad 1 — Renombrado y estandarización de columnas

*Dataset: Customer Personality Analysis*

Revisa los nombres de columna de `df_marketing` con `.columns`. Vas a notar una mezcla de convenciones reales: `Year_Birth` (con guion bajo), `Kidhome` (sin separador), `MntWines` (abreviado y sin separador).

**Trabaja sobre una copia** (`df_marketing_renombrado = df_marketing.copy()`) para no afectar las actividades siguientes, que usan los nombres originales. Aplica `.str.lower()` para al menos unificar mayúsculas/minúsculas, y usa `.rename()` para corregir manualmente los 2-3 nombres que la técnica automática no deja perfectos (por ejemplo, `mntwines` sigue sin ser ideal — decide tú el nombre final).

In [ ]:
# Tu código aquí
df_marketing_renombrado = df_marketing.copy()

# Estandarización automática: todo a minúsculas
df_marketing_renombrado.columns = df_marketing_renombrado.columns.str.lower()

# Correcciones manuales de los nombres que la técnica automática no deja ideales
df_marketing_renombrado = df_marketing_renombrado.rename(columns={
    'mntwines': 'monto_vinos',
    'mntfruits': 'monto_frutas',
    'mntmeatproducts': 'monto_carnes',
})

print(df_marketing_renombrado.columns.tolist())

['id', 'year_birth', 'education', 'marital_status', 'income', 'kidhome', 'teenhome', 'dt_customer', 'recency', 'monto_vinos', 'monto_frutas', 'monto_carnes', 'mntfishproducts', 'mntsweetproducts', 'mntgoldprods', 'numdealspurchases', 'numwebpurchases', 'numcatalogpurchases', 'numstorepurchases', 'numwebvisitsmonth', 'acceptedcmp3', 'acceptedcmp4', 'acceptedcmp5', 'acceptedcmp1', 'acceptedcmp2', 'complain', 'z_costcontact', 'z_revenue', 'response']


---
## Actividad 2 — Ajuste de tipos: fechas con formato mixto

*Dataset: Netflix*

La columna `date_added` de `df_netflix` mezcla formatos reales: la mayoría son `"14-Aug-20"`, pero un grupo minoritario llega como `" August 4, 2017"` (con espacio inicial). Conviértela a tipo fecha usando `pd.to_datetime(..., format='mixed')`, que resuelve ambos formatos en la misma columna. Verifica con `.dtypes` y confirma cuántos valores nulos quedan después de la conversión (compara contra los nulos que ya traía antes de convertir).

In [ ]:
# Tu código aquí
nulos_antes = df_netflix['date_added'].isnull().sum()

df_netflix['date_added'] = pd.to_datetime(df_netflix['date_added'], format='mixed')

nulos_despues = df_netflix['date_added'].isnull().sum()

print(df_netflix.dtypes['date_added'])
print('Nulos antes de convertir:', nulos_antes)
print('Nulos después de convertir:', nulos_despues)

datetime64[ns]
Nulos antes de convertir: 10
Nulos después de convertir: 10


---
## Actividad 3 — Duplicados

*Dataset: Customer Personality Analysis — con 2 filas duplicadas inyectadas a propósito*

Este dataset real no trae duplicados de forma natural — para poder practicar, se insertan 2 copias de clientes ya existentes (ejecuta la celda siguiente).

In [ ]:
df_marketing_dup = pd.concat([df_marketing, df_marketing.sample(2, random_state=7)], ignore_index=True)
print('Filas originales:', len(df_marketing))
print('Filas con duplicados inyectados:', len(df_marketing_dup))

Filas originales: 2240
Filas con duplicados inyectados: 2242


Sobre `df_marketing_dup`: cuenta los duplicados exactos con `.duplicated().sum()`, luego cuenta los duplicados por `ID` con `.duplicated(subset='ID').sum()` (deberían coincidir, ya que el `ID` es único por cliente). Elimínalos con `.drop_duplicates()` y confirma el número final de filas.

In [ ]:
# Tu código aquí
duplicados_exactos = df_marketing_dup.duplicated().sum()
duplicados_por_id = df_marketing_dup.duplicated(subset='ID').sum()

print('Duplicados exactos:', duplicados_exactos)
print('Duplicados por ID:', duplicados_por_id)

df_marketing_sin_dup = df_marketing_dup.drop_duplicates()
print('Filas finales tras eliminar duplicados:', len(df_marketing_sin_dup))

Duplicados exactos: 2
Duplicados por ID: 2
Filas finales tras eliminar duplicados: 2240


---
## Actividad 4 — Valores faltantes

*Dataset: Netflix*

Usa `.isnull().sum()` sobre `df_netflix` para ver cuántos valores faltan por columna. Luego, usa `.isnull().any(axis=1)` para filtrar solo las filas que tienen **al menos un** valor faltante en cualquier columna, y muestra cuántas filas son en total (`.sum()` sobre el resultado booleano).

In [ ]:
# Tu código aquí
nulos_por_columna = df_netflix.isnull().sum()
print(nulos_por_columna)

filas_con_nulos = df_netflix.isnull().any(axis=1)
print('Filas con al menos un valor faltante:', filas_con_nulos.sum())

show_id            0
type               0
title              0
director        2389
cast             718
country          507
date_added        10
release_year       0
rating             7
duration           0
listed_in          0
description        0
dtype: int64
Filas con al menos un valor faltante: 2979


---
## Actividad 5 — Completitud como porcentaje

*Dataset: Netflix*

Con los mismos nulos de la Actividad 4, calcula la completitud en porcentaje por columna: `(1 - nulos / total_filas) * 100`. ¿Qué columna tiene la completitud más baja? Escribe la respuesta en una línea.

In [ ]:
# Tu código aquí
total_filas = len(df_netflix)
completitud = (1 - df_netflix.isnull().sum() / total_filas) * 100
print(completitud)

print('Columna con menor completitud:', completitud.idxmin(), f"({completitud.min():.2f}%)")

show_id         100.000000
type            100.000000
title           100.000000
director         69.320663
cast             90.779504
country          93.489149
date_added       99.871581
release_year    100.000000
rating           99.910107
duration        100.000000
listed_in       100.000000
description     100.000000
dtype: float64
Columna con menor completitud: director (69.32%)


---
## Actividad 6 — Exploración categórica

*Dataset: Customer Personality Analysis*

Aplica `.value_counts()` sobre la columna `Marital_Status` de `df_marketing`. Vas a encontrar, junto a las categorías esperadas (`Married`, `Single`, `Together`, `Divorced`, `Widow`), tres valores que claramente son errores de captura reales: `Alone`, `Absurd` y `YOLO`. Decide y justifica en una línea: ¿los eliminarías, los reclasificarías (por ejemplo, `Alone` → `Single`), o los dejarías así? No hay una única respuesta correcta — lo que importa es la justificación.

In [ ]:
# Tu código aquí
print(df_marketing['Marital_Status'].value_counts())

# Decisión: reclasificaría "Alone" como "Single" (es sinónimo, no un error real),
# y eliminaría o marcaría como categoría "Otro/Error" los valores "Absurd" y "YOLO",
# porque no describen un estado civil real y probablemente vienen de una broma o error de captura.

Marital_Status
Married     864
Together    580
Single      480
Divorced    232
Widow        77
Alone         3
Absurd        2
YOLO          2
Name: count, dtype: int64


---
## Actividad 7 — Consistencia de formato/patrón

*Dataset: Netflix*

La columna `show_id` debería seguir siempre el patrón: la letra `s` seguida de uno o más dígitos (`s1`, `s2`, ..., `s8807`). Verifica con `.str.match(r'^s\d+$')` si todos los valores cumplen esta convención. Reporta el porcentaje de cumplimiento.

In [ ]:
# Tu código aquí
cumple_patron = df_netflix['show_id'].str.match(r'^s\d+$')
porcentaje_cumplimiento = cumple_patron.mean() * 100

print('Valores que cumplen el patrón:', cumple_patron.sum(), 'de', len(df_netflix))
print(f'Porcentaje de cumplimiento: {porcentaje_cumplimiento:.2f}%')

Valores que cumplen el patrón: 7787 de 7787
Porcentaje de cumplimiento: 100.00%


---
## Actividad 8 — `.info()` y `.describe()`

*Dataset: Customer Personality Analysis*

Ejecuta `.describe()` sobre la columna `Year_Birth` de `df_marketing` (puedes hacerlo con `df_marketing[['Year_Birth']].describe()`). Observa el valor mínimo (`min`). ¿Tiene sentido ese año de nacimiento? Filtra el DataFrame para mostrar las filas con los años de nacimiento más antiguos y decide, en una línea, si los considerarías un error de captura.

In [ ]:
# Tu código aquí
print(df_marketing[['Year_Birth']].describe())

anio_minimo = df_marketing['Year_Birth'].min()
print('Año de nacimiento mínimo:', anio_minimo)

mas_antiguos = df_marketing[df_marketing['Year_Birth'] < 1920]
print(mas_antiguos[['ID', 'Year_Birth', 'Education', 'Marital_Status']])

# No tiene sentido: un año de nacimiento como 1893, 1899 o 1900 implicaría edades
# de más de 120 años en el momento de la encuesta, así que sí los considero errores de captura.

        Year_Birth
count  2240.000000
mean   1968.805804
std      11.984069
min    1893.000000
25%    1959.000000
50%    1970.000000
75%    1977.000000
max    1996.000000
Año de nacimiento mínimo: 1893
        ID  Year_Birth Education Marital_Status
192   7829        1900  2n Cycle       Divorced
239  11004        1893  2n Cycle         Single
339   1150        1899       PhD       Together


---
## Actividad 9 — Práctica integradora: checklist de profiling

*Dataset: muestra real de Customer Personality Analysis, con 2 elementos inyectados y marcados a propósito (una fila duplicada y un valor de tipo incorrecto) para poder practicar el checklist completo con un resultado garantizado.*

Aplica el checklist completo, en orden, sobre `df_practica`:

1. Revisa `.dtypes` e identifica qué columna tiene un problema de tipo, corrígela con `pd.to_numeric(errors='coerce')`
2. Cuenta las filas duplicadas y elimínalas con `.drop_duplicates()`
3. Cuenta los valores faltantes por columna con `.isnull().sum()` (incluyendo el que se generó en el paso 1)
4. Revisa `.unique()` sobre `Marital_Status` y decide si necesita normalización

Al final, escribe un breve "reporte de profiling" (3-4 líneas) resumiendo qué encontraste y qué decidiste.

In [ ]:
# Muestra real con 2 elementos inyectados (marcados abajo)
df_practica = df_marketing.sample(15, random_state=3).reset_index(drop=True).copy()

# Elemento inyectado 1: una fila duplicada
df_practica = pd.concat([df_practica, df_practica.iloc[[2]]], ignore_index=True)

# Elemento inyectado 2: un valor de tipo incorrecto en Income
df_practica['Income'] = df_practica['Income'].astype(object)
df_practica.loc[5, 'Income'] = 'sesenta mil'

df_practica[['ID', 'Marital_Status', 'Income']]

,ID,Marital_Status,Income
0,5788,Together,46053.0
1,7930,Single,26877.0
2,4557,Together,22070.0
3,9964,Single,61825.0
4,1168,Married,72159.0
5,5314,Together,sesenta mil
6,9665,Divorced,54237.0
7,6182,Together,26646.0
8,922,Married,31086.0
9,4427,Single,83257.0


In [ ]:
# Paso 1 — ajuste de tipos
print(df_practica.dtypes)
# La columna Income tiene un problema de tipo: quedó como object por el valor "sesenta mil"

df_practica['Income'] = pd.to_numeric(df_practica['Income'], errors='coerce')
print(df_practica.dtypes['Income'])
print(df_practica[['ID', 'Income']])

ID                      int64
Year_Birth              int64
Education              object
Marital_Status         object
Income                 object
Kidhome                 int64
Teenhome                int64
Dt_Customer            object
Recency                 int64
MntWines                int64
MntFruits               int64
MntMeatProducts         int64
MntFishProducts         int64
MntSweetProducts        int64
MntGoldProds            int64
NumDealsPurchases       int64
NumWebPurchases         int64
NumCatalogPurchases     int64
NumStorePurchases       int64
NumWebVisitsMonth       int64
AcceptedCmp3            int64
AcceptedCmp4            int64
AcceptedCmp5            int64
AcceptedCmp1            int64
AcceptedCmp2            int64
Complain                int64
Z_CostContact           int64
Z_Revenue               int64
Response                int64
dtype: object
float64
       ID   Income
0    5788  46053.0
1    7930  26877.0
2    4557  22070.0
3    9964  61825.0
4    1168  72

In [ ]:
# Paso 2 — duplicados
duplicados_practica = df_practica.duplicated().sum()
print('Filas duplicadas encontradas:', duplicados_practica)

df_practica = df_practica.drop_duplicates()
print('Filas después de eliminar duplicados:', len(df_practica))

Filas duplicadas encontradas: 1
Filas después de eliminar duplicados: 15


In [ ]:
# Paso 3 — valores faltantes
print(df_practica.isnull().sum())

ID                     0
Year_Birth             0
Education              0
Marital_Status         0
Income                 1
Kidhome                0
Teenhome               0
Dt_Customer            0
Recency                0
MntWines               0
MntFruits              0
MntMeatProducts        0
MntFishProducts        0
MntSweetProducts       0
MntGoldProds           0
NumDealsPurchases      0
NumWebPurchases        0
NumCatalogPurchases    0
NumStorePurchases      0
NumWebVisitsMonth      0
AcceptedCmp3           0
AcceptedCmp4           0
AcceptedCmp5           0
AcceptedCmp1           0
AcceptedCmp2           0
Complain               0
Z_CostContact          0
Z_Revenue              0
Response               0
dtype: int64


In [ ]:
# Paso 4 — exploración categórica
print(df_practica['Marital_Status'].unique())
# En esta muestra las categorías de Marital_Status son todas válidas (Together, Single,
# Married, Divorced), así que no se necesita normalización en este subconjunto.

['Together' 'Single' 'Married' 'Divorced']


**Tu reporte de profiling:**

Sobre la muestra de 16 filas de `df_practica`, encontré un problema de tipo en la columna
`Income` (un valor de texto "sesenta mil" mezclado con números), que corregí convirtiéndola
con `pd.to_numeric(errors='coerce')`; esto generó 1 valor nulo donde antes había un dato
inválido. También había 1 fila duplicada exacta (mismo `ID` 4557 repetido), que eliminé con
`.drop_duplicates()`, dejando 15 filas únicas. En cuanto a `Marital_Status`, todas las
categorías presentes en esta muestra (Together, Single, Married, Divorced) son válidas y no
requirieron normalización. En general, la muestra estaba en buen estado salvo por estos dos
problemas puntuales, ambos ya identificados y corregidos.